# MQR: bounded batching and profiling diagnostics

Attach the current private readiness dataset and enable a T4 accelerator. This notebook reuses the canonical verified-source setup and tokenizer bootstrap. It spends at most one hour by default (a two-hour hard configuration ceiling) on isolated training-cost workers and separate profiler workers. It writes no training checkpoints and never starts a 3B-token run.

The full 12-layer, d512, context-512 backbone and actual verified 64k tokenizer remain unchanged. The global batch is always 32 sequences (16384 tokens) per successful update. Test microbatches 1,2,4,8,16,32 with accumulation 32,16,8,4,2,1. Two repeats reverse dense/MQR execution order. Each worker starts from the same seed, samples one common global batch and slices identical windows in original order; the complete LR schedule remains unchanged. Stop larger shared candidates after any worker OOM or failure. An OOM in one head does not establish that the other head would also OOM.

This run measures the current implementation, including the semantics-preserving unit-temperature and single-query score fast paths added after the first calibration. Batch 1 is measured again here. Compare against this fresh implementation baseline before attributing changes to batching alone. No initialization or architecture change is introduced by those fast paths.


In [ ]:
from pathlib import Path
import importlib.metadata
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile

# Normally autodetected. Set a mounted directory or ZIP path only if ambiguous.
UPLOAD_ROOT = None
CODE_ROOT = Path("/kaggle/working/mqr-tuning-source")
assert sys.version_info >= (3, 11), "Python 3.11+ is required"
if UPLOAD_ROOT is None:
    mounted = Path("/kaggle/input")
    candidates = list(mounted.rglob("mqr-kaggle-readiness.zip"))
    candidates += [marker.parent for marker in mounted.rglob("pyproject.toml")
                   if (marker.parent / "src" / "mqr").is_dir()]
    if len(candidates) != 1:
        raise ValueError("Set UPLOAD_ROOT to the intended source directory or ZIP; found: "
                         + repr([str(path) for path in candidates]))
    UPLOAD_ROOT = candidates[0]
else:
    UPLOAD_ROOT = Path(UPLOAD_ROOT)
assert UPLOAD_ROOT.exists(), f"Source upload does not exist: {UPLOAD_ROOT}"
print("Selected upload:", UPLOAD_ROOT)
if not CODE_ROOT.exists():
    archives = [UPLOAD_ROOT] if UPLOAD_ROOT.is_file() else list(UPLOAD_ROOT.glob("*.zip"))
    if archives:
        assert len(archives) == 1, "Choose the exact source ZIP using UPLOAD_ROOT"
        CODE_ROOT.mkdir(parents=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.infolist():
                target = (CODE_ROOT / item.filename).resolve()
                assert target.is_relative_to(CODE_ROOT.resolve()), "ZIP path escapes source directory"
                assert (item.external_attr >> 16) & 0o170000 != 0o120000, "ZIP symlinks are not supported"
            archive.extractall(CODE_ROOT)
    else:
        shutil.copytree(UPLOAD_ROOT, CODE_ROOT)
else:
    print("Reusing copied source; choose a fresh CODE_ROOT when changing the uploaded ZIP")
projects = list(CODE_ROOT.rglob("pyproject.toml"))
assert len(projects) == 1, "Source upload must contain one project pyproject.toml"
PROJECT = projects[0].parent
package_manifest = PROJECT / "package_manifest.json"
if package_manifest.exists():
    payload = json.loads(package_manifest.read_text(encoding="utf-8"))
    for name, expected in payload["files"].items():
        artifact = (PROJECT / name).resolve()
        assert artifact.is_relative_to(PROJECT.resolve()), "Manifest path escapes project"
        digest = hashlib.sha256()
        with artifact.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1048576), b""):
                digest.update(chunk)
        assert artifact.stat().st_size == expected["bytes"], f"Package size mismatch: {name}"
        assert digest.hexdigest() == expected["sha256"], f"Package hash mismatch: {name}"
    print("Verified packaged payload files:", len(payload["files"]))
# Use the existing CUDA torch; do not resolve or reinstall project dependencies.
assert int(importlib.metadata.version("setuptools").split(".")[0]) >= 77, "Install setuptools>=77 separately, then rerun setup"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(PROJECT)])
# Editable .pth files are processed at startup, so expose source to this live kernel.
sys.path.insert(0, str(PROJECT / "src"))
print("Installed source:", PROJECT)
print("Included prepared data:", PROJECT / "prepared")


def require_tokenizers():
    """Bootstrap only the tokenizer wheel before any tokenizers import."""
    def supported(value):
        try:
            version = tuple(map(int, value.split(".")[:2]))
        except (AttributeError, TypeError, ValueError):
            return False
        return len(version) == 2 and (0, 21) <= version < (0, 23)

    try:
        installed = importlib.metadata.version("tokenizers")
    except importlib.metadata.PackageNotFoundError:
        installed = None
    loaded = sys.modules.get("tokenizers")
    if loaded is not None and getattr(loaded, "__version__", None) != installed:
        raise RuntimeError("A different tokenizers version is already loaded; restart the clean kernel before setup")
    if not supported(installed):
        if any(name == "tokenizers" or name.startswith("tokenizers.") for name in sys.modules):
            raise RuntimeError("tokenizers is already imported; restart the clean kernel before changing its wheel")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps",
                               "--only-binary=:all:", "tokenizers==0.22.2"])
        importlib.invalidate_caches()
        installed = importlib.metadata.version("tokenizers")
        if installed != "0.22.2":
            raise RuntimeError("Tokenizer installation did not expose the requested tokenizers==0.22.2 wheel")
    if not supported(installed):
        raise RuntimeError("Prepared BPE data requires tokenizers>=0.21,<0.23")
    return installed

# Run before importing MQR or any library that could load tokenizers.
print("Tokenizers runtime:", require_tokenizers())


In [ ]:
from pathlib import Path
import json
import time
from mqr.kaggle import inspect_device
from mqr.tuning import run_sweep

DEVICE = "cuda:0"  # One GPU per subprocess; the second visible T4 is not used.
DATA_ROOT = PROJECT / "prepared"
print(json.dumps(inspect_device(DEVICE), indent=2))
RUN_ROOT = Path("/kaggle/working") / ("mqr-tuning-" + time.strftime("%Y%m%d-%H%M%S"))
print("Verified data location:", DATA_ROOT)
print("Outputs:", RUN_ROOT)


## Run the bounded sweep and separate profiles

Each configuration uses three successful warmup updates and eight measured updates; all attempts, AMP retries, exact full-vocabulary losses, window hashes, synchronized durations and allocated/reserved peaks are preserved. Only zero-overflow configurations with matching windows, both order repeats, and peak reserved allocator memory at most 90% of physical GPU capacity are eligible. This leaves a stated 10% headroom margin; driver and other-process allocations remain outside that allocator measurement. Timing includes common batch sampling/hash, transfers and the entire optimizer attempt; it excludes worker setup, log writes and profiler instrumentation.

Subprocess isolation clears model/optimizer allocations between configurations and recovers cleanly after OOM. A wall-clock timeout can also truncate the grid; inspect the stop reason and never treat an incomplete grid as a measured OOM boundary. Profiling reserves separate budget and selects the common eligible batch with the highest geometric mean throughput across both heads/repeats. The full trace contains one global optimizer update. The head-only trace contains one selected microbatch of exact vocabulary CE forward/backward, with detached backbone features, normal 64-position chunking/checkpointing, and no optimizer update. Their workload sizes differ and are reported; profiler times are not sweep throughput.

Unexpected trial errors/timeouts or failed/incomplete profiles make the protocol incomplete. The notebook preserves an archive first, then fails explicitly; these outcomes are not successful OOM boundaries. Trace retention is capped at 100 MiB per head worker (200 MiB total); oversized complete traces are dropped and labeled, while profiler key averages remain available.


In [ ]:
RUN_TUNING = True
if RUN_TUNING:
    result = run_sweep(data_dir=DATA_ROOT, output_root=RUN_ROOT, device=DEVICE,
                       batches=(1, 2, 4, 8, 16, 32), repeats=2,
                       warmup_steps=3, measured_steps=8,
                       budget_seconds=3600, worker_timeout=240, profiles=True)
    print(json.dumps({"status": result["status"], "best_shared_batch": result["best_shared_batch"],
                      "stop_reason": result["stop_reason"],
                      "total_wall_seconds": result["total_wall_seconds"],
                      "jobs": [{"batch": job["batch_size"], "head": job["head"],
                                "repeat": job["repeat"], "status": job["status"],
                                "tokens_s": job["summary"].get("weighted_tokens_per_second"),
                                "timing_eligible": job["summary"].get("timing_eligible")}
                               for job in result["jobs"]]}, indent=2))
    archive_path = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT)
    print("Preserved diagnostics archive:", archive_path)
    if not result["protocol_completed"]:
        raise RuntimeError("Tuning protocol incomplete; inspect archived summary/logs before proceeding: " + result["status"])


## Preserve and interpret results

Download the archive below. It includes raw per-attempt metrics, configs, verified data/source identities, worker logs, the paired sweep summary, and separate profiler key averages/Chrome traces. No large model checkpoint is generated. Inspect CUDA-time-recorded flags before claiming a CUDA profile was captured.

These short fresh-start updates do not measure convergence, ANN decoding or language-model quality. Low allocated memory alone does not establish GPU underutilization; use the traces to examine launch gaps and kernel costs. Treat the best shared microbatch as a configuration-specific performance choice, and preserve both repeats and failed workers when reporting it. The larger learning-rate warmup schedule is intentionally intact.


In [ ]:
if RUN_TUNING and "archive_path" in globals():
    print("Download from Kaggle Files:", archive_path)
